# P107 — Dapper, una infraestructura de trazado de sistemas distribuidos a gran escala

## 1. Título y paper

**Paper:** *Dapper, a Large-Scale Distributed Systems Tracing Infrastructure*  
**Autoría:** Benjamin H. Sigelman, Luiz André Barroso, Michael Burrows, Pat Stephenson, Manoj Plakal, Donald Beaver, Saul Jaspan, Chandan Shanbhag  
**Año y venue:** 2010 · Google Technical Report  
**Nivel:** L2 · **Motor:** `trazas_distribuidas`  
**Ficha completa:** [`P107_dapper`](../../papers/foundational/P107_dapper/README.md)

**Hito:** Hace observable una petición que atraviesa decenas de servicios, con un identificador que viaja con ella y un muestreo que la hace asequible.

- [Informe técnico de Google](https://research.google/pubs/pub36356/)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: En una arquitectura distribuida, cada servicio tiene sus métricas y sus registros. Cuando una petición va lenta, nadie puede reconstruir por dónde pasó ni dónde se gastó el tiempo: se ve el total y nada más.
2. Ejecutar una implementación mínima de la propuesta: Propagar un identificador de traza con la petición por todos los servicios, registrar un span por operación con su relación padre-hijo, y muestrear una fracción de las trazas para que el coste sea asumible sin perder los agregados.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P109


## 4. Intuición

Una petición tarda dos segundos. ¿Dónde se fueron? Cada servicio tiene sus propias métricas y todas dicen que están bien. Sin un identificador que viaje **con la petición**, nadie puede reconstruir su historia.


## 5. Concepto mínimo

```text
Span: una operación con inicio, fin, servicio y padre
Traza: el árbol de spans de UNA petición, unido por un identificador que viaja con ella

    puerta → autenticación → catálogo → recomendador → base_de_datos

Muestreo: guardar 1 de cada N trazas. Barato, y suficiente para los agregados.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('trazas_distribuidas', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué se puede diagnosticar viendo solo el tiempo total?
2. ¿Y con la traza desglosada por servicio?
3. ¿Se pierde precisión al muestrear el 1 % de las trazas?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('trazas_distribuidas', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('trazas_distribuidas', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con el total solo se sabe que el p50 es 230,71 ms y el p99 1 147 ms — «el sistema a veces va lento», que no es un diagnóstico. Con traza, el **recomendador** se lleva el 75,4 % del tiempo y tiene un p99 de 1 078 ms frente a un p50 de 158,88. Y muestreando el 1 % de las trazas, la estimación del p50 se desvía **5,12 ms** del valor real.


## 10. Comentario pedagógico

El muestreo es la decisión que hace viable el sistema y también su punto ciego: estima bien los agregados y **pierde los casos raros**. Si el fallo ocurre en una de cada mil peticiones, muestrear al 1 % probablemente no lo captura. Por eso hoy se usa muestreo dirigido por cola: decidir si guardar la traza **después** de ver si fue lenta o falló.


## 11. Error o anti-patrón deliberado

Anti-patrón: montar métricas por servicio y llamarlo observabilidad.


In [ ]:
print('Cada servicio con sus metricas responde: «yo estoy bien».')
print('Y la peticion del usuario tarda dos segundos.')
print('Sin identificador de traza propagado, no hay forma de unir la historia de UNA peticion.')

## 12. Corrección

Lo que la traza permite y las métricas no:


In [ ]:
r = run_paper_lab('trazas_distribuidas', seed=7)['result']
print('sin traza :', r['sin_traza'])
print()
for s in r['con_traza_por_servicio']:
    print(f"  {s['servicio']:<18} p50={s['p50']:<8} p99={s['p99']:<9} cuota={s['cuota_del_total']} %")

## 13. Desafío guiado

Compara el p50 y el p99 del recomendador y explica por qué la diferencia entre ambos es la señal que importa.


In [ ]:
r = run_paper_lab('trazas_distribuidas', seed=3)['result']
show(r)

## 14. Desafío autónomo

Instrumenta con trazado distribuido una cadena de dos o tres servicios de tu trabajo y localiza dónde se va el tiempo del p99. Después decide qué tasa de muestreo usarías y justifícala.


## 15. Evidencia de aprendizaje

Guarda el desglose por servicio con su cuota del total y tu decisión de tasa de muestreo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P107_dapper/README.md) · evaluación formal: [`assessments/papers/P107_dapper.md`](../../assessments/papers/P107_dapper.md)


## 16. Cierre

Ya se ve dónde se va el tiempo. La siguiente pregunta es qué hacer cuando una parte del sistema deja de responder.


## 17. Conexión con el siguiente hito

- P117

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
